In [16]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *
from test_utils import *

In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [19]:
datasetName = "Heloc"

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

dataset = pd.read_csv(f"{ugce_dir}/data/heloc.csv")
dataset = dataset[(dataset.iloc[:, 1:] >= 0).all(axis=1)]
dataset = dataset.reset_index(drop=True)
first_column = dataset.pop(dataset.columns[0])
TARGET_COLUMN = "RiskPerformance"
dataset[TARGET_COLUMN] = first_column
dataset[TARGET_COLUMN] = LabelEncoder().fit_transform(dataset[TARGET_COLUMN])
target = dataset[TARGET_COLUMN]

datasetX = dataset.copy()
datasetX = datasetX.drop(columns=[TARGET_COLUMN])

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.columns.to_list()
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])

    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])

    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances
print("Number of instances to explain: ", len(instances_to_explain))

Accuracy:  0.688622754491018
Number of instances to explain:  368


In [21]:
len(numerical), len(categorical)

(23, 0)

* df.iloc[0]: retrieves based on an index iterator from start to bottom.
* df.loc[0]: retrieves based on the index the df has.

# Calculate Stats

In [22]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

In [40]:
feat_unique = {}
for col in datasetX.columns:
    feat_unique[col] = len(datasetX[col].unique())
feat_unique
top_3_difficult_to_change_cols = sorted(feat_unique.items(), key=lambda x: x[1])
# top_3_difficult_to_change_cols += ['age']
print(f"Top 3 difficult to change columns: {top_3_difficult_to_change_cols[:3]}")
features_to_vary = [col for col in datasetX.columns if col not in top_3_difficult_to_change_cols]
features_to_vary

Top 3 difficult to change columns: [('MaxDelqEver', 5), ('MaxDelq2PublicRecLast12M', 8), ('NumBank2NatlTradesWHighUtilization', 12)]


['ExternalRiskEstimate',
 'MSinceOldestTradeOpen',
 'MSinceMostRecentTradeOpen',
 'AverageMInFile',
 'NumSatisfactoryTrades',
 'NumTrades60Ever2DerogPubRec',
 'NumTrades90Ever2DerogPubRec',
 'PercentTradesNeverDelq',
 'MSinceMostRecentDelq',
 'MaxDelq2PublicRecLast12M',
 'MaxDelqEver',
 'NumTotalTrades',
 'NumTradesOpeninLast12M',
 'PercentInstallTrades',
 'MSinceMostRecentInqexcl7days',
 'NumInqLast6M',
 'NumInqLast6Mexcl7days',
 'NetFractionRevolvingBurden',
 'NetFractionInstallBurden',
 'NumRevolvingTradesWBalance',
 'NumInstallTradesWBalance',
 'NumBank2NatlTradesWHighUtilization',
 'PercentTradesWBalance']

In [49]:
import warnings
import time
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

import dice_ml
d = dice_ml.Data(dataframe=dataset, continuous_features=numerical, outcome_name=TARGET_COLUMN)
backend = 'sklearn'
m = dice_ml.Model(model=model, backend=backend)

explainers = []
for i in range(5):
    exp_genetic = dice_ml.Dice(d, m, method='genetic')
    dice_exp_genetic = exp_genetic.generate_counterfactuals(
        instances_to_explain, total_CFs=2, desired_class="opposite",
        features_to_vary=features_to_vary)
    explainers.append(dice_exp_genetic)
import os
import pickle
results_dir = f'{UGCE_dir}/results/dice_objects/{datasetName}'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(explainers, open(f'{results_dir}/dice_exp_genetic.pkl', 'wb'))

100%|██████████| 368/368 [00:48<00:00,  7.59it/s]


In [23]:
# load explainers
import pickle
results_dir = f'{UGCE_dir}/results/dice_objects/{datasetName}'
explainers = pickle.load(open(f'{results_dir}/dice_exp_genetic.pkl', 'rb'))

In [ ]:
aggregate_results_DICE_baseline(iea, instances_to_explain, explainers, TARGET_COLUMN)

Full Time: mean = 32.1243, std = 2.7245
Generations: mean = 6.9065, std = 1.0674
Coverage: mean = 100.0000, std = 0.0000
Proximity Loss: mean = 0.0642, std = 0.0004
Sparsity: mean = 0.0295, std = 0.0000


# UGCE

## From Scratch

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

constraints = {
    'MaxDelqEver': '-',
    'MaxDelq2PublicRecLast12M': '-',
    'NumBank2NatlTradesWHighUtilization': '-'
}

import time
start_time = time.time()
strategy = "fix_population_update_fitness"

results_baseline_explainer = []
for i in range(5):
    results_baseline = iea.explain_instances(negative_instances, seed_number=None,
        dynamic_constraints=False, constraints=constraints,
        initial_population_variability=0.7, data_distribution=True,
        strategy=strategy, population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=20, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints={}, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric='weighted_l1',
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    time_taken_from_scratch = time.time() - start_time
    results_baseline_explainer.append(results_baseline)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/baseline'
os.makedirs(results_dir, exist_ok=True)
strategy = "fix_population_update_fitness"
pickle.dump(results_baseline_explainer, open(f'{results_dir}/results_baseline{strategy}.pkl', 'wb'))

100%|██████████| 368/368 [00:37<00:00,  9.80it/s]


In [ ]:
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/baseline'
results_baseline_explainer = pickle.load(open(f'{results_dir}/results_baselinefix_population_update_fitness.pkl', 'rb'))

In [13]:
from test_utils import *
aggregate_results_baseline(iea, results_baseline_explainer)

Full Time: mean = 36.5222, std = 0.2204
Generations: mean = 6.0000, std = 0.0000
Coverage: mean = 99.4595, std = 0.0000
Proximity Loss: mean = 0.0331, std = 0.0000
Sparsity: mean = 0.0288, std = 0.0000


## Dynamic

In [56]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'MaxDelqEver': 'i',
    'MaxDelq2PublicRecLast12M': 'i',
    'NumBank2NatlTradesWHighUtilization': 'i',
    
    'ExternalRiskEstimate': '',
    'MSinceOldestTradeOpen': '',
    'MSinceMostRecentTradeOpen': '',
    'AverageMInFile': '',
    'NumSatisfactoryTrades': '',
    'NumTrades60Ever2DerogPubRec': '',
    'NumTrades90Ever2DerogPubRec': '',
    'PercentTradesNeverDelq': '',
    'MSinceMostRecentDelq': '',
    'Max_Delq2PublicRecLast12M': '',
    'NumTotalTrades': '',
    'NumTradesOpeninLast12M': '',
    'PercentInstallTrades': '',
    'MSinceMostRecentInqexcl7days': '',
    'NumInqLast6M': '',
    'NumInqLast6Mexcl7days': '',
    'NetFractionRevolvingBurden': '',
    'NetFractionInstallBurden': '',
    'NumRevolvingTradesWBalance': '',
    'NumInstallTradesWBalance': '',
    'PercentTradesWBalance': '',
}

results_incremental_explainer = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer, open(f'{results_dir}/results_incremental{strategy}.pkl', 'wb'))

100%|██████████| 368/368 [01:08<00:00,  5.41it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:08<00:00,  5.36it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:07<00:00,  5.45it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:09<00:00,  5.31it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:08<00:00,  5.34it/s]


Empty intermediate counter: 0


In [57]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer, verbose=True)

Full Time: mean = 31.24, std = 0.34
Generations: mean = 6.00, std = 0.00
Coverage: mean = 92.43, std = 0.00
Proximity Loss: mean = 0.03, std = 0.00
Sparsity: mean = 0.03, std = 0.00
Intermediate Best Distances: mean = 0.05, std = 0.00


(31.238273000717165,
 6.0,
 92.43243243243244,
 0.033462419249193714,
 0.028913651488519605,
 0.04520221378779129)

# Assess the statistical Importance of correcting the population rather than starting with a random population

## Fixed population

In [58]:
total_time_dynamic_arr_for_ttest=[]
total_generations_arr_for_ttest=[]
total_cfes_found_arr_for_ttest=[]
total_proximity_loss_arr_for_ttest=[]
total_sparsity_arr_for_ttest=[]
total_best_intermediate_best_dist_arr_for_ttest=[]

full_times = []
coverages = []
distances = []
l1s = []
proximities = []
sparsities = []
generation_counts = []
intermediate_best_distances = []

for explainer in results_incremental_explainer:
    forttest, foravgs = stats_incremental(iea, explainer, return_matrices=True)
    time_dynamic_arr, generations_arr, cfes_found_arr, proximity_loss_arr, sparsity_arr, best_intermediate_best_dist_arr = forttest
    time_dynamic, avg_generations, avg_cfes_found, avg_l2, avg_l1, avg_proximity_loss, avg_sparsity, avg_best_intermediate_best_dist = foravgs
    
    total_time_dynamic_arr_for_ttest.extend(time_dynamic_arr)
    total_generations_arr_for_ttest.extend(generations_arr)
    total_cfes_found_arr_for_ttest.extend(cfes_found_arr)
    total_proximity_loss_arr_for_ttest.extend(proximity_loss_arr)
    total_sparsity_arr_for_ttest.extend(sparsity_arr)
    total_best_intermediate_best_dist_arr_for_ttest.extend(best_intermediate_best_dist_arr)

    full_times.append(time_dynamic)
    coverages.append(avg_cfes_found)
    distances.append(avg_l2)
    l1s.append(avg_l1)
    proximities.append(avg_proximity_loss)
    sparsities.append(avg_sparsity)
    generation_counts.append(avg_generations)
    intermediate_best_distances.append(avg_best_intermediate_best_dist)

def print_metric_stats(name, values):
        print(f"{name}: mean = {np.mean(values):.4f}, std = {np.std(values):.4f}")

## Random population

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'MaxDelqEver': 'i',
    'MaxDelq2PublicRecLast12M': 'i',
    'NumBank2NatlTradesWHighUtilization': 'i',
    
    'ExternalRiskEstimate': '',
    'MSinceOldestTradeOpen': '',
    'MSinceMostRecentTradeOpen': '',
    'AverageMInFile': '',
    'NumSatisfactoryTrades': '',
    'NumTrades60Ever2DerogPubRec': '',
    'NumTrades90Ever2DerogPubRec': '',
    'PercentTradesNeverDelq': '',
    'MSinceMostRecentDelq': '',
    'Max_Delq2PublicRecLast12M': '',
    'NumTotalTrades': '',
    'NumTradesOpeninLast12M': '',
    'PercentInstallTrades': '',
    'MSinceMostRecentInqexcl7days': '',
    'NumInqLast6M': '',
    'NumInqLast6Mexcl7days': '',
    'NetFractionRevolvingBurden': '',
    'NetFractionInstallBurden': '',
    'NumRevolvingTradesWBalance': '',
    'NumInstallTradesWBalance': '',
    'PercentTradesWBalance': '',
}

results_incremental_explainer_random = []
for i in range(5):
    import time
    strategy = "new_random_population"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="new_random_population", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_random.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_random, open(f'{results_dir}/results_incremental{strategy}.pkl', 'wb'))

100%|██████████| 368/368 [01:52<00:00,  3.28it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:51<00:00,  3.30it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:18<00:00,  4.71it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:16<00:00,  4.81it/s]


Empty intermediate counter: 0


100%|██████████| 368/368 [01:15<00:00,  4.90it/s]


Empty intermediate counter: 0


In [ ]:
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_explainer_random = pickle.load(open(f'{results_dir}/results_incrementalnew_random_population.pkl', 'rb'))

In [31]:
total_time_dynamic_arr_for_ttest_random=[]
total_generations_arr_for_ttest_random=[]
total_cfes_found_arr_for_ttest_random=[]
total_proximity_loss_arr_for_ttest_random=[]
total_sparsity_arr_for_ttest_random=[]
total_best_intermediate_best_dist_arr_for_ttest_random=[]

full_times_random = []
coverages_random = []
distances_random = []
l1s_random = []
proximities_random = []
sparsities_random = []
generation_counts_random = []
intermediate_best_distances_random = []

for explainer in results_incremental_explainer_random:
    forttest, foravgs = stats_incremental(iea, explainer, return_matrices=True)
    time_dynamic_arr, generations_arr, cfes_found_arr, proximity_loss_arr, sparsity_arr, best_intermediate_best_dist_arr = forttest
    time_dynamic, avg_generations, avg_cfes_found, avg_l2, avg_l1, avg_proximity_loss, avg_sparsity, avg_best_intermediate_best_dist = foravgs
    
    total_time_dynamic_arr_for_ttest_random.extend(time_dynamic_arr)
    total_generations_arr_for_ttest_random.extend(generations_arr)
    total_cfes_found_arr_for_ttest_random.extend(cfes_found_arr)
    total_proximity_loss_arr_for_ttest_random.extend(proximity_loss_arr)
    total_sparsity_arr_for_ttest_random.extend(sparsity_arr)
    total_best_intermediate_best_dist_arr_for_ttest_random.extend(best_intermediate_best_dist_arr)

    full_times_random.append(time_dynamic)
    coverages_random.append(avg_cfes_found)
    distances_random.append(avg_l2)
    l1s_random.append(avg_l1)
    proximities_random.append(avg_proximity_loss)
    sparsities_random.append(avg_sparsity)
    generation_counts_random.append(avg_generations)
    intermediate_best_distances_random.append(avg_best_intermediate_best_dist)

def print_metric_stats(name, values):
        print(f"{name}: mean = {np.mean(values):.4f}, std = {np.std(values):.4f}")

In [60]:
from scipy.stats import ttest_ind
import numpy as np

from scipy.stats import ttest_ind
import numpy as np

def format_stat(value):
    """Format a statistic to 2 decimal places"""
    return f"{value:.2f}"


def format_pvalue(pvalue):
    """Format p-value with scientific notation"""
    if not np.isfinite(pvalue):
        return "N/A"
    if pvalue < 1e-100:
        return "0.0\\times 10^{0}"
    
    try:
        sci_notation = f"{pvalue:.1e}".split('e')
        base = float(sci_notation[0])
        exponent = int(sci_notation[1])
        return f"{base:.1f}\\times 10^{{{exponent}}}"
    except Exception:
        return "N/A"


# Perform t-tests 
ttest_times = ttest_ind(total_time_dynamic_arr_for_ttest, total_time_dynamic_arr_for_ttest_random)
ttest_generations = ttest_ind(total_generations_arr_for_ttest, total_generations_arr_for_ttest_random)
ttest_cfes_found = ttest_ind(total_cfes_found_arr_for_ttest, total_cfes_found_arr_for_ttest_random)
ttest_distances = ttest_ind(total_proximity_loss_arr_for_ttest, total_proximity_loss_arr_for_ttest_random)
ttest_sparsity = ttest_ind(total_sparsity_arr_for_ttest, total_sparsity_arr_for_ttest_random)
test_distances_intermediate = ttest_ind(total_best_intermediate_best_dist_arr_for_ttest, total_best_intermediate_best_dist_arr_for_ttest_random)

warm_means = [
    np.mean(full_times),
    np.mean(generation_counts), 
    np.mean(coverages),
    np.mean(proximities),
    np.mean(sparsities),
    # np.mean(intermediate_best_distances)
]

random_means = [
    np.mean(full_times_random),
    np.mean(generation_counts_random),
    np.mean(coverages_random),
    np.mean(proximities_random),
    np.mean(sparsities_random),
    # np.mean(intermediate_best_distances_random),
]

pvalues = [
    ttest_times.pvalue,
    ttest_generations.pvalue, 
    ttest_cfes_found.pvalue,
    ttest_distances.pvalue,
    ttest_sparsity.pvalue,
    # test_distances_intermediate.pvalue
]

rows = [
    f"\\shortstack{{\\texttt{{Compas}}}} & {' & '.join(f'${format_pvalue(p)}$' for p in pvalues)} \\\\",
    f"Warm & {' & '.join(f'${format_stat(m)}$' for m in warm_means)} \\\\",
    f"Random & {' & '.join(f'${format_stat(m)}$' for m in random_means)} \\\\"
]

print("\n".join(rows))

\shortstack{\texttt{Compas}} & $0.0\times 10^{0}$ & $N/A$ & $7.5\times 10^{-1}$ & $0.0\times 10^{0}$ & $0.0\times 10^{0}$ \\
Warm & $31.24$ & $6.00$ & $92.43$ & $0.03$ & $0.03$ \\
Random & $41.51$ & $6.00$ & $92.16$ & $0.16$ & $0.02$ \\
